### 1. Setup and configuration
All knobs are in `CFG`. `P7_CFG_OVERRIDE` (JSON in an env var) overrides them without editing the cell.

In [1]:

# Setup and configuration

import os, sys, json, re, math, time, csv, random, zipfile, shutil, hashlib, warnings, gc, types, subprocess
from dataclasses import dataclass, asdict, fields
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np

def _ensure(pip_name, import_name):
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

_ensure("tokenizers", "tokenizers")
_ensure("huggingface_hub", "huggingface_hub")
_ensure("scikit-learn", "sklearn")
_ensure("pandas", "pandas")

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint

warnings.filterwarnings("ignore", category=UserWarning)

# Configuration
# Optional quick test: set P7_CFG_OVERRIDE before running.

CFG = dict(
    EXP_NAME="exp1_model2_lora_docpool",
    P7_TAG="p7_exp1_model2_lora_docpool_v1",          # tag stored in this notebook's checkpoints
    P1_TAG="decoder_only_rope_gqa_swiglu_clean_bpe32k_v1",   # tag of the Phase-1 Model-2 checkpoint

    # where things are (None = auto-discover under /kaggle/input)
    P1_CKPT=None,               # path to the Phase-1 Model-2 best.pt
    P1_TOKENIZER_JSON=None,     # path to the Phase-1 BPE tokenizer.json
    P7_ROOT=None,               # folder that contains Phase_7_PCR/ (None = download from Hugging Face)
    HF_DATASET_REPO="Exploration-Lab/CS-781-Capstone",

    # document representation
    STRIP_PLACEHOLDERS=True,    # remove <NAME>, <ORG>, <CITATION> ... anonymisation tags from the text
    MAX_CHARS=100_000,          # characters kept per case (head of the document)
    CHUNK_LEN=1024,             # Phase-1 context length; must be <= the Phase-1 max_len (1088)
    CHUNK_STRIDE=1024,          # == CHUNK_LEN -> no overlap between chunks
    MIN_TAIL=64,                # a trailing chunk shorter than this is dropped
    MAX_CHUNKS=16,              # chunks per case at inference (cap: 16 x 1024 tokens)
    TOKEN_POOL="mean",          # inside a chunk:  mean | last
    DOC_POOL="mean",            # across chunks:   mean | max | first | attn
    CENTER="none",              # none | pool  (subtract the evaluated pool's mean embedding)
    PROJ_DIM=None,              # None = d_model (identity-initialised projection head)
    TFIDF_NGRAM=(1, 1),         # (1, 2) uses a lot more RAM

    # LoRA
    LORA_R=16, LORA_ALPHA=32, LORA_DROPOUT=0.05,
    LORA_TARGETS=("q_proj", "k_proj", "v_proj", "o_proj", "w_gate", "w_up", "w_down"),

    # training
    TRAIN=True,
    EPOCHS=8,
    BATCH_Q=8,                  # queries per step (each brings 1 positive + HARD_NEGS hard negatives)
    HARD_NEGS=2,
    HARD_POOL=30,               # hard negatives are sampled from the TF-IDF top-HARD_POOL (gold removed)
    TRAIN_CHUNKS=2,             # random chunks per document at train time
    TEMP=0.05,                  # InfoNCE temperature
    LR=2e-4, HEAD_LR=5e-5, WEIGHT_DECAY=0.01, WARMUP_FRAC=0.1, MIN_LR_RATIO=0.1, GRAD_CLIP=1.0,
    GRAD_CKPT=True,
    TOK_BUDGET_TRAIN=16384,     # tokens per forward micro-batch while training (lower if CUDA OOM)
    TOK_BUDGET_EVAL=32768,      # tokens per forward micro-batch at inference
    EVAL_EVERY=1,               # dev evaluation every N epochs
    PATIENCE=3,                 # stop after N dev evaluations without improvement
    MAX_RUNTIME_H=10.5,

    # evaluation / output
    K=7,
    SEED=1234,
    RUN_ZERO_SHOT=True,         # analysis: Phase-1 model without any adaptation
    RUN_PREDICT=True,
    USE_HYBRID=False,           # True = final ranking = dense + lambda * TF-IDF (lambda picked on dev)
    MAX_QUERIES=None,           # smoke-test switches (per split)
    MAX_POOL=None,
)
_ov = os.environ.get("P7_CFG_OVERRIDE")
if _ov:
    CFG.update(json.loads(_ov))
CFG["LORA_TARGETS"] = tuple(CFG["LORA_TARGETS"])
CFG["TFIDF_NGRAM"] = tuple(CFG["TFIDF_NGRAM"])
C = types.SimpleNamespace(**CFG)
assert C.TOKEN_POOL in ("mean", "last") and C.DOC_POOL in ("mean", "max", "first", "attn") and C.CENTER in ("none", "pool")

# Paths and device

IS_KAGGLE = os.path.isdir("/kaggle/working")
INPUT_ROOT = "/kaggle/input"
BASE_DIR = "/kaggle/working" if IS_KAGGLE else "."
WORK_DIR = os.path.join(BASE_DIR, "p7_" + C.EXP_NAME)
CKPT_DIR = os.path.join(WORK_DIR, "checkpoints")
OUT_DIR = os.path.join(BASE_DIR, "cs781_phase7_" + C.EXP_NAME)       # prediction.json + submission.zip land here
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = {"enabled": False, "dtype": torch.float16}
if DEVICE.type == "cuda":
    AMP["enabled"] = True
    AMP["dtype"] = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16

def amp_ctx():
    return torch.autocast(device_type="cuda", dtype=AMP["dtype"], enabled=AMP["enabled"])

def log(*a):
    print(*a, flush=True)

def fmt_time(sec):
    sec = int(max(sec, 0))
    return f"{sec // 3600}h{(sec % 3600) // 60:02d}m{sec % 60:02d}s"

def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def walk_find(filename, roots):
    """all directories under `roots` that contain a file called `filename`"""
    hits, seen = [], set()
    for root in roots:
        if not root or not os.path.isdir(root):
            continue
        for d, _, files in os.walk(root):
            if filename in files and d not in seen:
                seen.add(d)
                hits.append(d)
    return hits

def read_json(path):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None

def atomic_json(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=1)
    os.replace(tmp, path)

def atomic_torch_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)

def hf_token():

    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret("HF")
        if t:
            return t
    except Exception:
        pass
    return os.environ.get("HF_TOKEN") or os.environ.get("HF") or None

HF_TOKEN = hf_token()
set_seed(C.SEED)
RESULTS = {"config": {k: (list(v) if isinstance(v, tuple) else v) for k, v in CFG.items()}}
T_START = time.time()

log(f"torch {torch.__version__} | device {DEVICE} | "
    f"{torch.cuda.get_device_name() if DEVICE.type == 'cuda' else 'NO GPU (this will be very slow)'} | "
    f"autocast {AMP['dtype'] if AMP['enabled'] else 'off'}")
log(f"HF token available: {bool(HF_TOKEN)} | work dir {WORK_DIR} | output dir {OUT_DIR}")

torch 2.11.0+cu130 | device cuda | Tesla T4 | autocast torch.float16
HF token available: True | work dir /kaggle/working/p7_exp1_model2_lora_docpool | output dir /kaggle/working/cs781_phase7_exp1_model2_lora_docpool


### 2. Phase-1 Model 2 and tokenizer
The architecture classes are copied from the Phase-1 notebook so the checkpoint loads with `strict=True`.

In [2]:

# Load the Phase-1 model

@dataclass
class ModelConfig:
    vocab_size: int = 32_000
    d_model: int = 640
    n_layers: int = 8
    n_heads: int = 10
    n_kv_heads: int = 2
    ffn_hidden: int = 2048
    dropout: float = 0.2
    max_len: int = 1024 + 64
    pad_id: int = 0
    unk_id: int = 1
    eos_id: int = 2

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight).type_as(x)

def build_rope(head_dim, max_len, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    freqs = torch.outer(torch.arange(max_len).float(), inv_freq)
    return freqs.cos(), freqs.sin()          # each (max_len, head_dim/2)

def apply_rope(x, cos, sin):
    # x: (B, H, T, hd); cos/sin: (1 or B, 1, T, hd/2)
    x1, x2 = x.float().chunk(2, dim=-1)
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.type_as(x)

def repeat_kv(x, groups):
    if groups == 1:
        return x
    B, KV, T, hd = x.shape
    return x[:, :, None].expand(B, KV, groups, T, hd).reshape(B, KV * groups, T, hd)

class GQAttention(nn.Module):
    """causal grouped-query self-attention with RoPE (10 query heads share 2 key/value heads)"""

    def __init__(self, cfg):
        super().__init__()
        self.h = cfg.n_heads
        self.kv = cfg.n_kv_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.groups = self.h // self.kv
        self.q_proj = nn.Linear(cfg.d_model, self.h * self.hd, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.o_proj = nn.Linear(self.h * self.hd, cfg.d_model, bias=False)

    def _qkv(self, x):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        return q, k, v

    def forward(self, x, cos, sin):
        B, T, _ = x.shape
        q, k, v = self._qkv(x)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)
        k, v = repeat_kv(k, self.groups), repeat_kv(v, self.groups)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, -1))

class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.w_gate = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_up = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_down = nn.Linear(cfg.ffn_hidden, cfg.d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn = GQAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.ffn = SwiGLU(cfg)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, cos, sin):
        x = x + self.drop(self.attn(self.norm1(x), cos, sin))
        x = x + self.drop(self.ffn(self.norm2(x)))
        return x

class DecoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)       # also the (tied) output projection
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = RMSNorm(cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        cos, sin = build_rope(cfg.d_model // cfg.n_heads, cfg.max_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def _rope(self, T, pos_ids=None):
        if pos_ids is None:
            return self.rope_cos[:T][None, None], self.rope_sin[:T][None, None]
        return self.rope_cos[pos_ids][:, None], self.rope_sin[pos_ids][:, None]

def count_params(model, trainable_only=False):
    return sum(p.numel() for p in model.parameters() if (p.requires_grad or not trainable_only))

# Load the Phase-1 checkpoint

def find_phase1_ckpt():
    if C.P1_CKPT:
        assert os.path.isfile(C.P1_CKPT), f"P1_CKPT not found: {C.P1_CKPT}"
        return C.P1_CKPT
    roots = [INPUT_ROOT, os.getcwd()]
    best = None                                                    # (val_loss, path)
    for d in walk_find("best.pt", roots):
        meta = read_json(os.path.join(d, "best_meta.json")) or {}
        if meta.get("tag") != C.P1_TAG:
            continue
        key = meta.get("val_loss", 1e9)
        if best is None or key < best[0]:
            best = (key, os.path.join(d, "best.pt"))
    if best is None:                                               # fall back to a latest.pt of this model
        for d in walk_find("latest.pt", roots):
            meta = read_json(os.path.join(d, "latest_meta.json")) or {}
            if meta.get("tag") != C.P1_TAG:
                continue
            key = -meta.get("global_step", 0)
            if best is None or key < best[0]:
                best = (key, os.path.join(d, "latest.pt"))
    if best is None:
        raise FileNotFoundError(
            "Phase-1 Model-2 checkpoint not found. Attach the Phase-1 output (the folder holding best.pt and "
            f"best_meta.json with tag '{C.P1_TAG}') as a Kaggle input, or set P1_CKPT.")
    return best[1]

def find_tokenizer_json(vocab_size):
    if C.P1_TOKENIZER_JSON:
        assert os.path.isfile(C.P1_TOKENIZER_JSON), f"P1_TOKENIZER_JSON not found: {C.P1_TOKENIZER_JSON}"
        return C.P1_TOKENIZER_JSON
    hits = []
    for d in walk_find("tokenizer.json", [INPUT_ROOT, os.getcwd()]):
        meta = read_json(os.path.join(d, "meta.json")) or {}
        ok = 0 if meta.get("vocab_size") == vocab_size else 1       # prefer the dir whose meta.json matches
        hits.append((ok, d))
    if not hits:
        raise FileNotFoundError("tokenizer.json (Phase-1 byte-level BPE) not found. Attach the BPE dataset "
                                "or set P1_TOKENIZER_JSON.")
    hits.sort(key=lambda t: t[0])
    return os.path.join(hits[0][1], "tokenizer.json")

class P1Tokenizer:
    """Phase-1 byte-level BPE (tokenizer.json).  No special tokens are ever added: Phase 1 never saw EOS inside text."""

    def __init__(self, path):
        from tokenizers import Tokenizer
        self.path = path
        self.tk = Tokenizer.from_file(path)
        vocab = self.tk.get_vocab()
        self.vocab_size = self.tk.get_vocab_size()
        self.pad_id, self.unk_id, self.eos_id = vocab["<|pad|>"], vocab["<|unk|>"], vocab["<|endoftext|>"]

    def encode_batch(self, texts):
        return [e.ids for e in self.tk.encode_batch(texts, add_special_tokens=False)]

def load_phase1():
    ck_path = find_phase1_ckpt()
    log(f"Phase-1 checkpoint: {ck_path}")
    ck = torch.load(ck_path, map_location="cpu", weights_only=False)
    names = {f.name for f in fields(ModelConfig)}
    cfg = ModelConfig(**{k: v for k, v in (ck.get("config") or {}).items() if k in names})
    cfg.dropout = 0.0                                              # no dropout in the frozen base
    sd = {(k[len("_orig_mod."):] if k.startswith("_orig_mod.") else k): v for k, v in ck["model"].items()}
    model = DecoderLM(cfg)
    model.load_state_dict(sd, strict=True)                         # raises on ANY key / shape mismatch
    log(f"loaded Phase-1 weights strictly | tag={ck.get('tag')} | "
        f"{count_params(model) / 1e6:.1f}M params | d_model={cfg.d_model} layers={cfg.n_layers} "
        f"heads={cfg.n_heads}/{cfg.n_kv_heads} max_len={cfg.max_len}")
    return model, cfg, ck_path, ck.get("tag")

P1_MODEL, P1_CFG, P1_CKPT_PATH, P1_CKPT_TAG = load_phase1()
tok_path = find_tokenizer_json(P1_CFG.vocab_size)
TOK = P1Tokenizer(tok_path)
assert TOK.vocab_size == P1_CFG.vocab_size, (TOK.vocab_size, P1_CFG.vocab_size)
assert (TOK.pad_id, TOK.unk_id, TOK.eos_id) == (P1_CFG.pad_id, P1_CFG.unk_id, P1_CFG.eos_id), \
    "special-token ids of tokenizer and checkpoint disagree"
assert C.CHUNK_LEN <= P1_CFG.max_len, f"CHUNK_LEN {C.CHUNK_LEN} exceeds Phase-1 max_len {P1_CFG.max_len}"
log(f"tokenizer: {tok_path} | vocab={TOK.vocab_size} pad={TOK.pad_id} unk={TOK.unk_id} eos={TOK.eos_id}")
if P1_CKPT_TAG != C.P1_TAG:
    log(f"WARNING: checkpoint tag {P1_CKPT_TAG!r} != expected {C.P1_TAG!r} (weights did load, so the architecture matches)")
RESULTS["phase1"] = {"ckpt": P1_CKPT_PATH, "tag": P1_CKPT_TAG, "params": count_params(P1_MODEL)}

Phase-1 checkpoint: /kaggle/input/datasets/krishrkalgude/model2-prep/cs781_phase1_model2/checkpoints/best.pt
loaded Phase-1 weights strictly | tag=decoder_only_rope_gqa_swiglu_clean_bpe32k_v1 | 59.8M params | d_model=640 layers=8 heads=10/2 max_len=1088
tokenizer: /kaggle/input/datasets/krishkalg/bpe-prep/cs781_clean_bpe32k/tokenizer.json | vocab=32000 pad=0 unk=1 eos=2


### 3. Phase-7 data
Queries and candidates per split. Sentence lists are joined in order; anonymisation tags (`<NAME>`, `<ORG>`, `<CITATION>`) are stripped by default.

In [3]:

# Prepare the data
# Keep each split's candidate pool separate.

P7_FILES = {
    "train_queries": "Phase_7_PCR/Train/train.json",
    "train_candidates": "Phase_7_PCR/Train/train_candidates.json",
    "dev_queries": "Phase_7_PCR/Dev/dev.json",
    "dev_candidates": "Phase_7_PCR/Dev/dev_candidates.json",
    "test_queries": "Phase_7_PCR/Test/test.json",
    "test_candidates": "Phase_7_PCR/Test/test_candidates.json",
}

def _looks_like_p7(path):
    p = path.lower().replace("-", "_")
    return "phase_7" in p or "phase7" in p or "pcr" in p

def p7_file(rel):
    """local copy (P7_ROOT / Kaggle input) first, then the Hugging Face dataset repo"""
    base = os.path.basename(rel)
    if C.P7_ROOT:
        for cand in (Path(C.P7_ROOT) / rel, Path(C.P7_ROOT) / base):
            if cand.is_file():
                return str(cand)
        for d in walk_find(base, [C.P7_ROOT]):
            return os.path.join(d, base)
    for d in walk_find(base, [INPUT_ROOT]):                         # an attached copy of the dataset
        if _looks_like_p7(d):
            return os.path.join(d, base)
    err = None
    try:
        from huggingface_hub import hf_hub_download
        return hf_hub_download(repo_id=C.HF_DATASET_REPO, filename=rel, repo_type="dataset", token=HF_TOKEN)
    except Exception as e:
        err = e
    raise FileNotFoundError(f"cannot find {rel}: not under P7_ROOT / {INPUT_ROOT} and the HF download failed: {err!r}")

def unwrap_json(obj):
    if isinstance(obj, dict):
        for key in ("data", "records", "examples", "queries", "candidates"):
            if isinstance(obj.get(key), (list, dict)):
                return obj[key]
    return obj

def normalize_text(value):
    if value is None:
        return ""
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, list):                                     # sentence list -> one string, order preserved
        return " ".join(str(x).strip() for x in value if x is not None).strip()
    return str(value).strip()

def records_from_queries(obj):
    """-> list of (query_id, text, [relevant ids])"""
    rows = []
    if isinstance(obj, list):
        iterator = enumerate(obj)
    elif isinstance(obj, dict):
        iterator = [(None, obj)] if any(k in obj for k in ("id", "case_id", "query_id")) else obj.items()
    else:
        raise TypeError(f"queries must be list/dict, got {type(obj)}")
    for key, rec in iterator:
        if not isinstance(rec, dict):
            continue
        cid = rec.get("id", rec.get("case_id", rec.get("query_id", key)))
        text = rec.get("text", rec.get("document", rec.get("case_text", "")))
        rel = rec.get("relevant_candidates", rec.get("relevant_candidate_ids", [])) or []
        if isinstance(rel, str):
            rel = [rel]
        rows.append((str(cid), normalize_text(text), [str(x) for x in rel]))
    return rows

def records_from_candidates(obj):
    """-> dict candidate_id -> text"""
    out = {}
    if isinstance(obj, dict):
        iterable = [(None, obj)] if any(k in obj for k in ("id", "case_id", "text", "document")) else obj.items()
    elif isinstance(obj, list):
        iterable = [(None, r) for r in obj]
    else:
        raise TypeError(f"candidates must be list/dict, got {type(obj)}")
    for key, rec in iterable:
        if isinstance(rec, dict):
            cid = rec.get("id", rec.get("case_id", rec.get("candidate_id", key)))
            txt = rec.get("text", rec.get("document", rec.get("case_text", "")))
        else:
            cid, txt = key, rec
        if cid is None:
            raise ValueError("cannot infer a candidate id - inspect the raw candidate file")
        out[str(cid)] = normalize_text(txt)
    return out

_PLACEHOLDER = re.compile(r"<\s*[A-Z][A-Z_]{1,19}\s*>")             # <NAME> <ORG> <CITATION> ...

def prep_text(t):
    if C.STRIP_PLACEHOLDERS:
        t = _PLACEHOLDER.sub(" ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t[:C.MAX_CHARS]

def limit_split(sp, max_q, max_pool, seed):
    """smoke-test helper: first `max_q` queries; pool = their gold + their own ids + random fill up to `max_pool`"""
    rng = random.Random(seed)
    qids = sp.qids[:max_q] if max_q else list(sp.qids)
    pool = set(sp.cids)
    keep = set()
    for q in qids:
        keep |= (sp.gold[q] & pool)
        if q in pool:
            keep.add(q)
    rest = [c for c in sp.cids if c not in keep]
    rng.shuffle(rest)
    n_fill = len(rest) if not max_pool else max(max_pool - len(keep), 0)
    keep |= set(rest[:n_fill])
    sp.qids = qids
    sp.cids = [c for c in sp.cids if c in keep]
    sp.qtext = {q: sp.qtext[q] for q in qids}
    sp.gold = {q: sp.gold[q] for q in qids}
    sp.ctext = {c: sp.ctext[c] for c in sp.cids}
    return sp

def load_split(name):
    q_raw = unwrap_json(json.load(open(p7_file(P7_FILES[f"{name}_queries"]), encoding="utf-8")))
    c_raw = unwrap_json(json.load(open(p7_file(P7_FILES[f"{name}_candidates"]), encoding="utf-8")))
    qrows = records_from_queries(q_raw)
    cmap = records_from_candidates(c_raw)
    sp = types.SimpleNamespace(name=name)
    sp.qids = [q for q, _, _ in qrows]
    sp.qtext = {q: prep_text(t) for q, t, _ in qrows}
    sp.gold = {q: set(r) for q, _, r in qrows}
    sp.cids = list(cmap.keys())
    sp.ctext = {c: prep_text(t) for c, t in cmap.items()}
    if C.MAX_QUERIES or C.MAX_POOL:
        sp = limit_split(sp, C.MAX_QUERIES, C.MAX_POOL, C.SEED)
        log(f"[smoke] {name}: limited to {len(sp.qids)} queries / {len(sp.cids)} candidates")
    pool = set(sp.cids)
    sp.gold_in_pool = {q: sorted((sp.gold[q] & pool) - {q}) for q in sp.qids}
    return sp

SP = {name: load_split(name) for name in ("train", "dev", "test")}

rows = []
for name, sp in SP.items():
    pool = set(sp.cids)
    ng = [len(sp.gold[q]) for q in sp.qids]
    nl = [len(sp.qtext[q].split()) for q in sp.qids]
    cl = [len(t.split()) for t in sp.ctext.values()]
    mism = sum(1 for q in sp.qids if q in sp.ctext and sp.ctext[q] != sp.qtext[q])
    rows.append({
        "split": name, "queries": len(sp.qids), "pool": len(sp.cids),
        "query_in_pool": sum(q in pool for q in sp.qids), "text_differs": mism,
        "gold_mean": round(float(np.mean(ng)), 2) if ng else 0, "gold_min": min(ng) if ng else 0,
        "gold_max": max(ng) if ng else 0,
        "gold_not_in_pool": sum(len(sp.gold[q] - pool) for q in sp.qids),
        "no_gold_queries": sum(len(sp.gold[q]) == 0 for q in sp.qids),
        "q_words_med": int(np.median(nl)) if nl else 0, "c_words_med": int(np.median(cl)) if cl else 0,
        "c_words_p95": int(np.percentile(cl, 95)) if cl else 0,
    })
log(pd.DataFrame(rows).to_string(index=False))
log("(query_in_pool > 0 means a query's own case is also in the pool: it is excluded from its own ranking)")
assert all(len(SP[s].qids) > 0 and len(SP[s].cids) > C.K for s in SP), "a split is empty / its pool is smaller than K+1"
assert sum(len(SP["train"].gold_in_pool[q]) > 0 for q in SP["train"].qids) > 0, "no train query has a gold candidate in the pool"
assert any(len(SP["dev"].gold[q]) > 0 for q in SP["dev"].qids), "dev has no labels"
RESULTS["data"] = rows
sample_txt = next(iter(SP["train"].qtext.values()))
log("sample (first 300 chars of a prepared train query):", sample_txt[:300])

Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

dev.json:   0%|          | 0.00/5.87M [00:00<?, ?B/s]

Phase_7_PCR/Dev/dev_candidates.json: reconstructing file:   0%|          |  0.00B / 51.9MB            

Phase_7_PCR/Dev/dev_candidates.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/9.34M [00:00<?, ?B/s]

Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

split  queries  pool  query_in_pool  text_differs  gold_mean  gold_min  gold_max  gold_not_in_pool  no_gold_queries  q_words_med  c_words_med  c_words_p95
train      827  4320            827             0       6.42         0        52                 0               10         4444         3574        16937
  dev      118  1023            118             0       8.25         1        92                 0                0         4564         3970        17324
 test      237  1727            237             0       0.00         0         0                 0              237         4586         3863        17020
(query_in_pool > 0 means a query's own case is also in the pool: it is excluded from its own ranking)
sample (first 300 chars of a prepared train query): PETITIONER: Vs. RESPONDENT: ROOP SINGH RATHORE & OTHERS(and connected appeal) DATE OF JUDGMENT: 07/05/1963 BENCH: , S.K. BENCH: , S.K. SINHA, BHUVNESHWAR P.(CJ) DAYAL, RAGHUBAR AYYANGAR, MUDHOLKAR, J.R. CITATION: 1964 AIR 1545

### 4. Tokenisation and chunking of long cases

In [4]:

# Tokenisation and chunking
# Split each case into chunks that fit the model context.

def chunk_spans(n, L, stride, max_chunks, min_tail):
    """token spans [(start, end), ...] covering a document of n tokens"""
    if n <= L:
        return [(0, n)]
    spans, s = [], 0
    while s < n and len(spans) < max_chunks:
        e = min(s + L, n)
        if spans and e - s < min_tail:                              # drop a tiny trailing chunk
            break
        spans.append((s, e))
        if e == n:
            break
        s += stride
    return spans

def _selftest_chunk_spans():
    assert chunk_spans(10, 1024, 1024, 16, 64) == [(0, 10)]
    assert chunk_spans(1024, 1024, 1024, 16, 64) == [(0, 1024)]
    assert chunk_spans(2048, 1024, 1024, 16, 64) == [(0, 1024), (1024, 2048)]
    assert chunk_spans(2048 + 30, 1024, 1024, 16, 64) == [(0, 1024), (1024, 2048)]           # tail < 64 dropped
    assert chunk_spans(2048 + 100, 1024, 1024, 16, 64) == [(0, 1024), (1024, 2048), (2048, 2148)]
    assert len(chunk_spans(10**6, 1024, 1024, 16, 64)) == 16
    assert chunk_spans(3000, 1024, 512, 16, 64)[-1][1] == 3000                                # overlap: reaches the end
    return True

_selftest_chunk_spans()

MAX_TOKENS = C.CHUNK_LEN + (C.MAX_CHUNKS - 1) * C.CHUNK_STRIDE      # hard cap on tokens kept per document

class Doc:
    __slots__ = ("ids", "spans")

    def __init__(self, ids):
        if len(ids) == 0:
            ids = np.array([TOK.unk_id], dtype=np.int32)             # never feed an empty sequence
        self.ids = ids
        self.spans = chunk_spans(len(ids), C.CHUNK_LEN, C.CHUNK_STRIDE, C.MAX_CHUNKS, C.MIN_TAIL)

_TOK_CACHE = {}

def build_docs(texts, batch=128):
    keys = [hashlib.md5(t.encode("utf-8")).hexdigest() for t in texts]
    todo = {}
    for k, t in zip(keys, texts):
        if k not in _TOK_CACHE and k not in todo:
            todo[k] = t
    items = list(todo.items())
    for i in range(0, len(items), batch):
        part = items[i:i + batch]
        enc = TOK.encode_batch([t for _, t in part])
        for (k, _), ids in zip(part, enc):
            _TOK_CACHE[k] = Doc(np.asarray(ids[:MAX_TOKENS], dtype=np.int32))
    return [_TOK_CACHE[k] for k in keys]

t0 = time.time()
for name, sp in SP.items():
    sp.qdocs = build_docs([sp.qtext[q] for q in sp.qids])
    sp.cdocs = build_docs([sp.ctext[c] for c in sp.cids])
    sp.q_index = {q: i for i, q in enumerate(sp.qids)}
    sp.c_index = {c: i for i, c in enumerate(sp.cids)}
log(f"tokenised {len(_TOK_CACHE):,} unique documents in {fmt_time(time.time() - t0)}")

rows = []
for name, sp in SP.items():
    for role, docs in (("query", sp.qdocs), ("candidate", sp.cdocs)):
        ntok = np.array([len(d.ids) for d in docs])
        nch = np.array([len(d.spans) for d in docs])
        rows.append({"split": name, "role": role, "docs": len(docs), "tok_median": int(np.median(ntok)),
                     "tok_p95": int(np.percentile(ntok, 95)), "tok_max": int(ntok.max()),
                     "chunks_mean": round(float(nch.mean()), 2), "chunks_max": int(nch.max()),
                     "at_token_cap": int((ntok >= MAX_TOKENS).sum()), "total_chunks": int(nch.sum())})
log(pd.DataFrame(rows).to_string(index=False))
RESULTS["tokenisation"] = rows

tokenised 5,819 unique documents in 0h00m44s
split      role  docs  tok_median  tok_p95  tok_max  chunks_mean  chunks_max  at_token_cap  total_chunks
train     query   827        5309    16384    16384         6.84          16            70          5659
train candidate  4320        4251    16384    16384         6.01          16           360         25969
  dev     query   118        5378    16384    16384         6.99          16            13           825
  dev candidate  1023        4702    16384    16384         6.51          16           121          6656
 test     query   237        5427    16384    16384         6.94          16            24          1644
 test candidate  1727        4603    16384    16384         6.41          16           167         11074


### 5. Retrieval utilities (ranking, micro-P/R/F1@7, TF-IDF, hard-negative mining)
Pure numpy / scikit-learn. Includes small self-tests.

In [5]:

# Retrieval utilities

def l2n(x, eps=1e-8):
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), eps)

def mask_self(S, qids, cids):
    """-inf where a candidate is the query itself (queries also sit in their pool). Works on a copy."""
    S = S.astype(np.float32, copy=True)
    cidx = {c: i for i, c in enumerate(cids)}
    for i, q in enumerate(qids):
        j = cidx.get(q)
        if j is not None:
            S[i, j] = -np.inf
    return S

def rank_topk(S, qids, cids, k):
    """top-k candidate ids per query (best first), the query's own id excluded.  S: [n_queries, n_candidates]"""
    S = mask_self(S, qids, cids)
    k = min(k, S.shape[1] - 1)
    part = np.argpartition(-S, k - 1, axis=1)[:, :k]
    sc = np.take_along_axis(S, part, axis=1)
    order = np.argsort(-sc, axis=1, kind="stable")
    top = np.take_along_axis(part, order, axis=1)
    return {q: [cids[j] for j in top[i]] for i, q in enumerate(qids)}

def micro_prf(preds, gold, qids, k=7):
    """micro P / R / F1@k against the FULL gold set (no truncation, no padding)"""
    tp = n_pred = n_gold = 0
    for q in qids:
        p = list(preds[q][:k])
        g = gold[q]
        tp += len(set(p) & g)
        n_pred += len(p)
        n_gold += len(g)
    P = tp / n_pred if n_pred else 0.0
    R = tp / n_gold if n_gold else 0.0
    F1 = 2 * P * R / (P + R) if (P + R) > 0 else 0.0
    return {"P": P, "R": R, "F1": F1, "tp": tp, "n_pred": n_pred, "n_gold": n_gold}

def _selftest_metrics():
    preds = {"a": ["1", "2", "3", "4", "5", "6", "7"], "b": ["1", "2", "3", "4", "5", "6", "7"]}
    gold = {"a": {"1", "2", "9"}, "b": {"7"}}
    m = micro_prf(preds, gold, ["a", "b"], 7)
    assert (m["tp"], m["n_pred"], m["n_gold"]) == (3, 14, 4)
    assert abs(m["P"] - 3 / 14) < 1e-12 and abs(m["R"] - 3 / 4) < 1e-12
    assert abs(m["F1"] - 2 * (3 / 14) * (3 / 4) / (3 / 14 + 3 / 4)) < 1e-12
    S = np.array([[0.9, 0.1, 0.5, 0.8], [0.2, 0.3, 0.1, 0.95]], dtype=np.float32)
    top = rank_topk(S, ["x", "y"], ["x", "b", "c", "d"], 2)          # query "x" is in the pool at column 0
    assert top["x"] == ["d", "c"] and top["y"] == ["d", "b"], top
    return True

_selftest_metrics()

def first_hit_rank(S, qids, cids, gold):
    """1-based rank of the best-ranked gold candidate (0 if the query has no gold in the pool)"""
    S = mask_self(S, qids, cids)
    cidx = {c: i for i, c in enumerate(cids)}
    out = []
    for i, q in enumerate(qids):
        g = [cidx[c] for c in gold[q] if c in cidx and c != q]
        out.append(int((S[i] > S[i, g].max()).sum()) + 1 if g else 0)
    return out

def tfidf_scores(q_texts, c_texts):
    """classic TF-IDF cosine similarity, fitted on the pool only. [n_queries, n_candidates] float32"""
    from sklearn.feature_extraction.text import TfidfVectorizer

    def make(min_df, max_df):
        return TfidfVectorizer(sublinear_tf=True, ngram_range=C.TFIDF_NGRAM, min_df=min_df, max_df=max_df,
                               max_features=300_000, dtype=np.float32)

    try:
        vec = make(2, 0.9)
        X = vec.fit_transform(c_texts)
    except ValueError:                                              # tiny / degenerate pool (smoke runs)
        vec = make(1, 1.0)
        X = vec.fit_transform(c_texts)
    Q = vec.transform(q_texts)
    return (Q @ X.T).toarray().astype(np.float32)

def zscore_rows(S):
    return (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-8)

def hybrid_scores(S_dense, S_tfidf, lam):
    return zscore_rows(S_dense) + lam * zscore_rows(S_tfidf)

def mine_hard_negs(S_tfidf, qids, cids, gold, pool_n):
    """for every query: the top `pool_n` TF-IDF candidates that are neither the query itself nor gold"""
    S = mask_self(S_tfidf, qids, cids)
    cidx = {c: i for i, c in enumerate(cids)}
    out = {}
    for i, q in enumerate(qids):
        for g in gold[q]:
            j = cidx.get(g)
            if j is not None:
                S[i, j] = -np.inf
        n = min(pool_n, S.shape[1] - 1)
        idx = np.argpartition(-S[i], n - 1)[:n]
        idx = idx[np.argsort(-S[i, idx])]
        out[q] = [cids[j] for j in idx if np.isfinite(S[i, j])]
    return out

def _selftest_mining():
    cids = ["x", "g1", "n1", "n2", "n3"]
    S = np.array([[0.99, 0.95, 0.9, 0.5, 0.4]], dtype=np.float32)    # "x" is itself, "g1" is gold
    hn = mine_hard_negs(S, ["x"], cids, {"x": {"g1"}}, 2)
    assert hn["x"] == ["n1", "n2"], hn
    return True

_selftest_mining()

def per_query_table(S, sp, preds, k):
    ranks = first_hit_rank(S, sp.qids, sp.cids, sp.gold)
    rows = []
    for i, q in enumerate(sp.qids):
        g = sp.gold[q]
        hits = len(set(preds[q][:k]) & g)
        rows.append({"qid": q, "n_gold": len(g), "hits": hits,
                     "precision": hits / k, "recall": hits / len(g) if g else 0.0,
                     "first_hit_rank": ranks[i], "q_tokens": len(sp.qdocs[i].ids),
                     "q_chunks": len(sp.qdocs[i].spans)})
    return pd.DataFrame(rows)

### 6. LoRA, retrieval encoder and document pooling

In [6]:

# CELL 6 - LoRA, retrieval encoder, chunked feature extraction, document pooling

# Mask padding only during pooling.

class LoRALinear(nn.Module):
    """y = W x + (alpha/r) * B(A(dropout(x))).  B starts at 0, so the adapted model == Phase 1 at step 0."""

    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.r = r
        self.scale = alpha / r
        dev = base.weight.device
        self.A = nn.Parameter(torch.empty(r, base.in_features, device=dev))
        self.B = nn.Parameter(torch.zeros(base.out_features, r, device=dev))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale

def inject_lora(model, r, alpha, dropout, targets):
    n = 0
    for blk in model.blocks:
        for parent in (blk.attn, blk.ffn):
            for name in targets:
                lin = getattr(parent, name, None)
                if isinstance(lin, nn.Linear):
                    setattr(parent, name, LoRALinear(lin, r, alpha, dropout))
                    n += 1
    return n

class RetrievalEncoder(nn.Module):
    def __init__(self, lm, proj_dim=None):
        super().__init__()
        self.lm = lm
        d = lm.cfg.d_model
        p = proj_dim or d
        self.head = nn.Linear(d, p, bias=False)                     # identity at init -> zero-shot == plain pooling
        nn.init.eye_(self.head.weight)
        self.attn = nn.Linear(p, 1)                                 # chunk attention pooler (zero init = uniform mean)
        nn.init.zeros_(self.attn.weight)
        nn.init.zeros_(self.attn.bias)
        self.use_ckpt = False

    def hidden(self, ids):
        """final-norm hidden states [B, T, d] of the (LoRA-adapted) causal LM, before the LM head"""
        lm = self.lm
        cos, sin = lm._rope(ids.size(1))
        x = lm.embed(ids)
        for blk in lm.blocks:
            if self.use_ckpt and self.training and torch.is_grad_enabled():
                x = grad_checkpoint(blk, x, cos, sin, use_reentrant=False)
            else:
                x = blk(x, cos, sin)
        return lm.final_norm(x)

    def token_features(self, ids, lens, need):
        h = self.hidden(ids).float()
        out = {}
        if "mean" in need:
            m = (torch.arange(ids.size(1), device=ids.device)[None, :] < lens[:, None]).unsqueeze(-1).float()
            out["mean"] = (h * m).sum(1) / lens[:, None].float()
        if "last" in need:
            out["last"] = h[torch.arange(ids.size(0), device=ids.device), lens - 1]
        return out

def chunk_features(enc, chunks, need, budget, train=False, use_amp=True):
    """
    chunks: list of 1-D int arrays (each <= CHUNK_LEN).  Returns {pool: [N, d] float32} in the ORIGINAL chunk order.
    Chunks are sorted by length so micro-batches carry little padding; each micro-batch holds <= `budget` tokens.
    train=True keeps the autograd graph (features stay on the GPU), otherwise features are moved to the CPU.
    """
    n = len(chunks)
    lens = np.array([len(c) for c in chunks], dtype=np.int64)
    order = np.argsort(-lens, kind="stable")
    outs = {k: [] for k in need}
    i = 0
    while i < n:
        L = int(lens[order[i]])
        mb = max(1, budget // max(L, 1))
        sel = order[i:i + mb]
        ids = np.zeros((len(sel), L), dtype=np.int64)
        for r, j in enumerate(sel):
            ids[r, :lens[j]] = chunks[j]
        ids_t = torch.from_numpy(ids).to(DEVICE)
        lens_t = torch.from_numpy(lens[sel]).to(DEVICE)
        with (amp_ctx() if use_amp else torch.autocast(device_type="cuda", enabled=False)):
            feats = enc.token_features(ids_t, lens_t, need)
        for k in need:
            outs[k].append(feats[k] if train else feats[k].float().cpu())
        i += mb
    inv = np.empty(n, dtype=np.int64)
    inv[order] = np.arange(n)
    res = {}
    for k in need:
        cat = torch.cat(outs[k], dim=0)
        res[k] = cat[torch.from_numpy(inv).to(cat.device)]
    return res

def docs_to_chunks(docs, train_chunks=None, rng=None):
    """flatten documents into chunks.  seg[i] = index of the document that chunk i belongs to (non-decreasing)."""
    chunks, seg = [], []
    for di, d in enumerate(docs):
        spans = d.spans
        if train_chunks and len(spans) > train_chunks:              # training: a random subset of the chunks
            pick = np.sort(rng.choice(len(spans), train_chunks, replace=False))
            spans = [spans[p] for p in pick]
        for s, e in spans:
            chunks.append(d.ids[s:e])
            seg.append(di)
    return chunks, np.asarray(seg, dtype=np.int64)

def assemble(enc, feat, seg, n_docs, doc_pool):
    """chunk features [N, d] -> head -> L2 -> pool the chunks of every document -> L2.   Differentiable."""
    z = F.normalize(enc.head(feat.float()), dim=-1)
    p = z.size(1)
    if doc_pool == "first":
        counts = torch.bincount(seg, minlength=n_docs)
        starts = torch.cumsum(counts, 0) - counts
        out = z[starts]
    elif doc_pool == "max":
        idx = seg[:, None].expand(-1, p)
        out = z.new_full((n_docs, p), float("-inf")).scatter_reduce(0, idx, z, reduce="amax", include_self=False)
    elif doc_pool == "attn":
        s = enc.attn(z).squeeze(-1)
        w = torch.exp(s - s.max().detach())
        out = z.new_zeros(n_docs, p).index_add(0, seg, z * w[:, None])   # the per-doc denominator cancels in the L2 norm
    else:                                                           # "mean" (sum of unit vectors; L2 norm makes it a mean)
        out = z.new_zeros(n_docs, p).index_add(0, seg, z)
    return F.normalize(out, dim=-1)

def _selftest_assemble():
    """vectorised document pooling == a naive per-document loop"""
    g = torch.Generator().manual_seed(0)
    d = 8
    enc = types.SimpleNamespace(head=nn.Identity(), attn=nn.Linear(d, 1))
    nn.init.normal_(enc.attn.weight, std=0.5)
    feat = torch.randn(7, d, generator=g)
    seg = torch.tensor([0, 0, 0, 1, 2, 2, 3])                       # 4 docs with 3 / 1 / 2 / 1 chunks
    z = F.normalize(feat, dim=-1)
    for pool in ("mean", "max", "first", "attn"):
        got = assemble(enc, feat, seg, 4, pool)
        for di in range(4):
            zz = z[seg == di]
            if pool == "mean":
                ref = zz.mean(0)
            elif pool == "max":
                ref = zz.max(0).values
            elif pool == "first":
                ref = zz[0]
            else:
                w = torch.softmax(enc.attn(zz).squeeze(-1), 0)
                ref = (w[:, None] * zz).sum(0)
            ref = F.normalize(ref, dim=-1)
            assert torch.allclose(got[di], ref, atol=1e-5), (pool, di)
    return True

_selftest_assemble()
log("self-tests passed: chunk spans, metrics, hard-negative mining, document pooling")

def build_encoder():
    lm = P1_MODEL.to(DEVICE)
    for p in lm.parameters():
        p.requires_grad_(False)
    enc = RetrievalEncoder(lm, C.PROJ_DIM).to(DEVICE)
    enc.use_ckpt = bool(C.GRAD_CKPT)
    return enc

def trainable_state(enc):
    return {n: p.detach().cpu().clone() for n, p in enc.named_parameters() if p.requires_grad}

def load_trainable_state(enc, state):
    missing, unexpected = enc.load_state_dict(state, strict=False)
    assert not unexpected, f"unexpected keys in the LoRA checkpoint: {unexpected[:5]}"
    own = {n for n, p in enc.named_parameters() if p.requires_grad}
    assert own <= set(state), f"checkpoint lacks trainable params: {sorted(own - set(state))[:5]}"

@torch.inference_mode()
def probe_vectors(enc, docs, n=4, max_len=256):
    """first-chunk mean features of a few documents, in full fp32 (used for the LoRA no-op check)"""
    was = enc.training
    enc.eval()
    ch = [d.ids[:max_len] for d in docs[:n]]
    f = chunk_features(enc, ch, ("mean",), 8192, train=False, use_amp=False)["mean"]
    enc.train(was)
    return f

@torch.inference_mode()
def encode_split_raw(enc, sp, need=("mean", "last")):
    """one pass of the LM over queries + pool of a split (all chunks).  Keeps the raw token-pooled chunk features."""
    was = enc.training
    enc.eval()
    docs = sp.qdocs + sp.cdocs
    chunks, seg = docs_to_chunks(docs)
    t0 = time.time()
    feats = chunk_features(enc, chunks, need, C.TOK_BUDGET_EVAL, train=False)
    enc.train(was)
    return {"feats": feats, "seg": seg, "n_docs": len(docs), "nq": len(sp.qdocs),
            "seconds": time.time() - t0, "n_chunks": len(chunks)}

@torch.inference_mode()
def embed_from_raw(enc, raw, token_pool, doc_pool, center="none"):
    """-> (query embeddings, candidate embeddings) as float32 numpy arrays, L2-normalised"""
    f = raw["feats"][token_pool].to(DEVICE)
    seg = torch.from_numpy(raw["seg"]).to(DEVICE)
    E = assemble(enc, f, seg, raw["n_docs"], doc_pool).float().cpu().numpy()
    Qe, Ce = E[:raw["nq"]], E[raw["nq"]:]
    if center == "pool":                                            # unsupervised: statistics of the evaluated pool only
        mu = Ce.mean(0, keepdims=True)
        Qe, Ce = l2n(Qe - mu), l2n(Ce - mu)
    return Qe, Ce

def evaluate_dense(Qe, Ce, sp, k=None):
    k = k or C.K
    S = Qe @ Ce.T
    preds = rank_topk(S, sp.qids, sp.cids, k)
    return micro_prf(preds, sp.gold, sp.qids, k), preds, S

self-tests passed: chunk spans, metrics, hard-negative mining, document pooling


### 7. Baselines, zero-shot Phase 1, LoRA injection check, hard negatives

In [7]:

# Build the encoder

ENC = build_encoder()
N_BASE_PARAMS = count_params(ENC.lm)
dev = SP["dev"]

def dev_grid(enc, raw, sp, token_pools=("mean", "last"), doc_pools=("mean", "max", "first"), centers=("none", "pool")):
    rows = []
    for tp in token_pools:
        for dp in doc_pools:
            for ce in centers:
                Qe, Ce = embed_from_raw(enc, raw, tp, dp, ce)
                m, _, _ = evaluate_dense(Qe, Ce, sp)
                rows.append({"token_pool": tp, "doc_pool": dp, "center": ce,
                             "P@7": round(m["P"], 4), "R@7": round(m["R"], 4), "F1@7": round(m["F1"], 4)})
    return pd.DataFrame(rows).sort_values("F1@7", ascending=False).reset_index(drop=True)

# TF-IDF baseline on dev (classical, no pretrained model)
t0 = time.time()
TFIDF_DEV = tfidf_scores([dev.qtext[q] for q in dev.qids], [dev.ctext[c] for c in dev.cids])
m_tfidf = micro_prf(rank_topk(TFIDF_DEV, dev.qids, dev.cids, C.K), dev.gold, dev.qids, C.K)
log(f"[baseline] TF-IDF dev  P@7={m_tfidf['P']:.4f}  R@7={m_tfidf['R']:.4f}  F1@7={m_tfidf['F1']:.4f}  "
    f"({fmt_time(time.time() - t0)})")
RESULTS["tfidf_dev"] = m_tfidf

# zero-shot Phase-1 representation (no adaptation) + pooling grid
PROBE_DOCS = dev.cdocs

@torch.inference_mode()
def padding_check(enc, docs):
    """right-padded, length-sorted micro-batches must give the same chunk features as encoding every chunk alone"""
    ch = [d.ids[:L] for d, L in zip(docs[:4], (200, 37, 120, 64))]
    enc_was = enc.training
    enc.eval()
    batch = chunk_features(enc, ch, ("mean", "last"), 8192, train=False, use_amp=False)
    worst = 0.0
    for i, c in enumerate(ch):
        one = chunk_features(enc, [c], ("mean", "last"), 8192, train=False, use_amp=False)
        for k in ("mean", "last"):
            worst = max(worst, float((batch[k][i] - one[k][0]).abs().max()))
    enc.train(enc_was)
    assert worst < 1e-3, f"padding / ordering changes the chunk features (max abs diff {worst:.2e})"
    return worst

log(f"padding + ordering check passed (max abs diff {padding_check(ENC, PROBE_DOCS):.1e})")
V0 = probe_vectors(ENC, PROBE_DOCS)                                  # fp32 features BEFORE LoRA exists
if C.RUN_ZERO_SHOT:
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    raw0 = encode_split_raw(ENC, dev)
    log(f"[zero-shot] encoded dev: {raw0['n_docs']:,} docs / {raw0['n_chunks']:,} chunks in "
        f"{fmt_time(raw0['seconds'])} ({raw0['n_docs'] / max(raw0['seconds'], 1e-9):.1f} docs/s)")
    GRID0 = dev_grid(ENC, raw0, dev)
    log("[zero-shot] Phase-1 Model 2, dev, no adaptation (token pooling x document pooling x centering):")
    log(GRID0.to_string(index=False))
    RESULTS["zero_shot_grid"] = GRID0.to_dict("records")
    RESULTS["efficiency_zero_shot"] = {
        "dev_docs": raw0["n_docs"], "dev_chunks": raw0["n_chunks"], "seconds": raw0["seconds"],
        "docs_per_s": raw0["n_docs"] / max(raw0["seconds"], 1e-9),
        "peak_gpu_mem_gb": (torch.cuda.max_memory_allocated() / 2**30) if DEVICE.type == "cuda" else None}
    del raw0
    gc.collect()

# inject LoRA (B = 0  =>  the model is still exactly Phase 1)
n_inj = inject_lora(ENC.lm, C.LORA_R, C.LORA_ALPHA, C.LORA_DROPOUT, C.LORA_TARGETS)
for n_, p_ in ENC.named_parameters():
    p_.requires_grad_(n_.startswith("lm.") and (n_.endswith(".A") or n_.endswith(".B")) or n_.startswith("head."))
for p_ in ENC.attn.parameters():
    p_.requires_grad_(C.DOC_POOL == "attn")                          # only trained if the attention pooler is used
V1 = probe_vectors(ENC, PROBE_DOCS)
noop = float((V1 - V0).abs().max())
assert noop < 1e-5, f"LoRA injection changed the Phase-1 output (max abs diff {noop:.3e}) - it must be a no-op at init"
n_lora = sum(p.numel() for n_, p in ENC.named_parameters() if n_.startswith("lm.") and (n_.endswith(".A") or n_.endswith(".B")))
n_head = sum(p.numel() for n_, p in ENC.named_parameters() if p.requires_grad and not n_.startswith("lm."))
n_train = count_params(ENC, trainable_only=True)
assert n_train == n_lora + n_head and all(not p.requires_grad for n_, p in ENC.named_parameters() if n_.endswith(".base.weight"))
log(f"LoRA injected into {n_inj} linears (r={C.LORA_R}, alpha={C.LORA_ALPHA}); no-op check passed (diff {noop:.1e})")
log(f"params: Phase-1 base {N_BASE_PARAMS / 1e6:.2f}M (frozen) | LoRA {n_lora / 1e6:.3f}M | head/pooler {n_head / 1e6:.3f}M | "
    f"trainable {n_train / 1e6:.3f}M = {100 * n_train / (N_BASE_PARAMS + n_train):.2f}% of the model")
RESULTS["params"] = {"base": N_BASE_PARAMS, "lora": n_lora, "head_and_pooler": n_head, "trainable": n_train,
                     "total": count_params(ENC), "lora_modules": n_inj}

# TF-IDF hard negatives for the training queries
tr = SP["train"]
t0 = time.time()
TFIDF_TRAIN = tfidf_scores([tr.qtext[q] for q in tr.qids], [tr.ctext[c] for c in tr.cids])
m_tr = micro_prf(rank_topk(TFIDF_TRAIN, tr.qids, tr.cids, C.K), tr.gold, tr.qids, C.K)
HARD_NEGS_MAP = mine_hard_negs(TFIDF_TRAIN, tr.qids, tr.cids, tr.gold, C.HARD_POOL)
log(f"[train] TF-IDF F1@7={m_tr['F1']:.4f}; hard-negative pools mined for {len(HARD_NEGS_MAP)} queries "
    f"(median {int(np.median([len(v) for v in HARD_NEGS_MAP.values()]))} per query) in {fmt_time(time.time() - t0)}")
RESULTS["tfidf_train"] = m_tr
del TFIDF_TRAIN
gc.collect()

[baseline] TF-IDF dev  P@7=0.3995  R@7=0.3392  F1@7=0.3669  (0h00m03s)
padding + ordering check passed (max abs diff 1.2e-05)
[zero-shot] encoded dev: 1,141 docs / 7,481 chunks in 0h01m05s (17.5 docs/s)
[zero-shot] Phase-1 Model 2, dev, no adaptation (token pooling x document pooling x centering):
token_pool doc_pool center    P@7    R@7   F1@7
      mean     mean   pool 0.3354 0.2847 0.3079
      mean     mean   none 0.3257 0.2765 0.2991
      mean      max   pool 0.2760 0.2343 0.2535
      mean      max   none 0.2615 0.2220 0.2401
      mean    first   pool 0.2506 0.2127 0.2301
      mean    first   none 0.2494 0.2117 0.2290
      last     mean   pool 0.1259 0.1069 0.1156
      last     mean   none 0.0981 0.0832 0.0901
      last      max   pool 0.0436 0.0370 0.0400
      last      max   none 0.0266 0.0226 0.0245
      last    first   none 0.0254 0.0216 0.0233
      last    first   pool 0.0218 0.0185 0.0200
LoRA injected into 56 linears (r=16, alpha=32); no-op check passed (diff 0.0e

0

### 8. Training (resumable)

In [8]:


# Checkpoints are saved after each epoch.

LATEST, BEST = os.path.join(CKPT_DIR, "p7_latest.pt"), os.path.join(CKPT_DIR, "p7_best.pt")
LATEST_META, BEST_META = os.path.join(CKPT_DIR, "p7_latest_meta.json"), os.path.join(CKPT_DIR, "p7_best_meta.json")
LOG_CSV = os.path.join(CKPT_DIR, "p7_train_log.csv")

def _gscaler():
    enabled = AMP["enabled"] and AMP["dtype"] == torch.float16
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled, init_scale=2.0 ** 12)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=enabled, init_scale=2.0 ** 12)

def lr_scale(step, total, warm):
    if step < warm:
        return (step + 1) / max(warm, 1)
    prog = (step - warm) / max(1, total - warm)
    return C.MIN_LR_RATIO + (1 - C.MIN_LR_RATIO) * 0.5 * (1 + math.cos(math.pi * prog))

def embed_train(docs, rng):
    chunks, seg = docs_to_chunks(docs, C.TRAIN_CHUNKS, rng)
    f = chunk_features(ENC, chunks, (C.TOKEN_POOL,), C.TOK_BUDGET_TRAIN, train=True)[C.TOKEN_POOL]
    return assemble(ENC, f, torch.from_numpy(seg).to(DEVICE), len(docs), C.DOC_POOL)

def evaluate_dev_now():
    raw = encode_split_raw(ENC, SP["dev"], need=(C.TOKEN_POOL,))
    Qe, Ce = embed_from_raw(ENC, raw, C.TOKEN_POOL, C.DOC_POOL, C.CENTER)
    m, _, _ = evaluate_dense(Qe, Ce, SP["dev"])
    return m, raw["seconds"]

def find_resume():
    best = None
    for d in walk_find("p7_latest_meta.json", [CKPT_DIR, INPUT_ROOT]):
        meta = read_json(os.path.join(d, "p7_latest_meta.json"))
        if not meta or meta.get("tag") != C.P7_TAG or not os.path.exists(os.path.join(d, "p7_latest.pt")):
            continue
        if best is None or meta["epoch"] > best[1]["epoch"]:
            best = (d, meta)
    return best

def save_best(state, f1):
    atomic_torch_save({"trainable": trainable_state(ENC), "tag": C.P7_TAG, "dev_f1": f1, "epoch": state["epoch"]}, BEST)
    atomic_json({"tag": C.P7_TAG, "dev_f1": f1, "epoch": state["epoch"]}, BEST_META)

def run_training():
    tr = SP["train"]
    train_qs = [q for q in tr.qids if tr.gold_in_pool[q]]
    B = C.BATCH_Q
    steps_per_epoch = math.ceil(len(train_qs) / B)
    total_steps = steps_per_epoch * C.EPOCHS
    warm = max(1, int(C.WARMUP_FRAC * total_steps))
    log(f"train queries with gold in the pool: {len(train_qs)}/{len(tr.qids)} | {steps_per_epoch} steps/epoch x "
        f"{C.EPOCHS} epochs = {total_steps} steps | warm-up {warm}")

    lora_params = [p for n, p in ENC.named_parameters() if p.requires_grad and n.startswith("lm.")]
    head_params = [p for n, p in ENC.named_parameters() if p.requires_grad and not n.startswith("lm.")]
    opt = torch.optim.AdamW(
        [{"params": lora_params, "lr": C.LR, "base_lr": C.LR, "weight_decay": 0.0},
         {"params": head_params, "lr": C.HEAD_LR, "base_lr": C.HEAD_LR, "weight_decay": C.WEIGHT_DECAY}],
        betas=(0.9, 0.999))
    scaler = _gscaler()
    all_trainable = lora_params + head_params

    state = {"epoch": 0, "gs": 0, "best_f1": -1.0, "best_epoch": -1, "bad": 0, "history": []}
    prev = find_resume()
    if prev is not None:
        d, meta = prev
        pl = torch.load(os.path.join(d, "p7_latest.pt"), map_location="cpu", weights_only=False)
        load_trainable_state(ENC, pl["trainable"])
        opt.load_state_dict(pl["opt"])
        scaler.load_state_dict(pl["scaler"])
        state = pl["state"]
        if os.path.abspath(d) != os.path.abspath(CKPT_DIR) and os.path.exists(os.path.join(d, "p7_best.pt")):
            shutil.copy(os.path.join(d, "p7_best.pt"), BEST)
            shutil.copy(os.path.join(d, "p7_best_meta.json"), BEST_META)
        log(f"RESUMED from {d}: {state['epoch']} epochs done, best dev F1@7 {state['best_f1']:.4f} (epoch {state['best_epoch']})")

    if state["best_f1"] < 0:                                          # epoch 0 = Phase 1 + untouched adapters
        m0, _ = evaluate_dev_now()
        state.update(best_f1=m0["F1"], best_epoch=0)
        save_best({"epoch": 0}, m0["F1"])
        state["history"].append({"epoch": 0, "train_loss": None, "dev_P": m0["P"], "dev_R": m0["R"], "dev_F1": m0["F1"]})
        log(f"epoch 0 (zero-shot, LoRA at init)  dev P@7={m0['P']:.4f} R@7={m0['R']:.4f} F1@7={m0['F1']:.4f}")

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t_train0, last_epoch_s, good_steps = time.time(), 0.0, 0
    docs_seen, t_docs = 0, 0.0

    for epoch in range(state["epoch"], C.EPOCHS):
        if (time.time() - T_START) / 3600 + 1.3 * last_epoch_s / 3600 > C.MAX_RUNTIME_H:
            log("time budget would be exceeded by another epoch -> stopping (state is saved, re-run to continue)")
            break
        t_ep = time.time()
        ENC.train()
        rng = np.random.default_rng(C.SEED + 1000 * epoch)
        perm = rng.permutation(len(train_qs))
        run_loss, run_acc, run_n = 0.0, 0.0, 0
        for step in range(steps_per_epoch):
            t_step = time.time()
            qb = [train_qs[i] for i in perm[step * B:(step + 1) * B]]
            cand_ids, cand_docs, seen, tgt = [], [], {}, []

            def add(c):
                if c not in seen:
                    seen[c] = len(cand_ids)
                    cand_ids.append(c)
                    cand_docs.append(tr.cdocs[tr.c_index[c]])
                return seen[c]

            for q in qb:                                              # one gold positive per query
                g = tr.gold_in_pool[q]
                tgt.append(add(g[int(rng.integers(len(g)))]))
            if C.HARD_NEGS:
                for q in qb:                                          # TF-IDF hard negatives
                    hn = HARD_NEGS_MAP[q]
                    if hn:
                        for t in rng.choice(len(hn), min(C.HARD_NEGS, len(hn)), replace=False):
                            add(hn[int(t)])
            mask = np.zeros((len(qb), len(cand_ids)), dtype=bool)     # other gold cases + the query itself are not negatives
            for i, q in enumerate(qb):
                for j, c in enumerate(cand_ids):
                    if j != tgt[i] and (c == q or c in tr.gold[q]):
                        mask[i, j] = True

            q_docs = [tr.qdocs[tr.q_index[q]] for q in qb]
            Qe = embed_train(q_docs, rng)
            Ce = embed_train(cand_docs, rng)
            logits = (Qe @ Ce.T) / C.TEMP
            logits = logits.masked_fill(torch.from_numpy(mask).to(DEVICE), -1e4)
            tgt_t = torch.tensor(tgt, device=DEVICE)
            loss = F.cross_entropy(logits, tgt_t)

            f = lr_scale(state["gs"], total_steps, warm)
            for gr in opt.param_groups:
                gr["lr"] = gr["base_lr"] * f
            if not torch.isfinite(loss):
                log(f"  non-finite loss at step {state['gs']}, batch skipped")
                opt.zero_grad(set_to_none=True)
                state["gs"] += 1
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            gnorm = float(torch.nn.utils.clip_grad_norm_(all_trainable, C.GRAD_CLIP))
            if math.isfinite(gnorm) and gnorm > 0:
                good_steps += 1
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            state["gs"] += 1
            if state["gs"] == 25 and good_steps == 0:
                raise RuntimeError("25 steps without a finite non-zero gradient - the LoRA graph is detached or fp16 overflows")

            run_loss += float(loss)
            run_acc += float((logits.argmax(1) == tgt_t).float().mean())
            run_n += 1
            docs_seen += len(q_docs) + len(cand_docs)
            t_docs += time.time() - t_step
            if (step + 1) % 10 == 0 or step + 1 == steps_per_epoch:
                log(f"[ep {epoch + 1}/{C.EPOCHS}] step {step + 1}/{steps_per_epoch} | loss {run_loss / max(run_n, 1):.4f} | "
                    f"in-batch acc {run_acc / max(run_n, 1):.3f} | gnorm {gnorm:.2f} | lr {opt.param_groups[0]['lr']:.2e} | "
                    f"{docs_seen / max(t_docs, 1e-9):.1f} docs/s | elapsed {fmt_time(time.time() - T_START)}")

        last_epoch_s = time.time() - t_ep
        state["epoch"] = epoch + 1
        row = {"epoch": epoch + 1, "train_loss": run_loss / max(run_n, 1), "train_acc": run_acc / max(run_n, 1),
               "epoch_seconds": last_epoch_s}
        if (epoch + 1) % C.EVAL_EVERY == 0 or epoch + 1 == C.EPOCHS:
            m, eval_s = evaluate_dev_now()
            row.update(dev_P=m["P"], dev_R=m["R"], dev_F1=m["F1"], dev_eval_seconds=eval_s)
            improved = m["F1"] > state["best_f1"] + 1e-6
            if improved:
                state.update(best_f1=m["F1"], best_epoch=epoch + 1, bad=0)
                save_best(state, m["F1"])
            else:
                state["bad"] += 1
            log(f"== epoch {epoch + 1}: train loss {row['train_loss']:.4f} | dev P@7={m['P']:.4f} R@7={m['R']:.4f} "
                f"F1@7={m['F1']:.4f} {'(new best)' if improved else ''} | best {state['best_f1']:.4f} @ epoch {state['best_epoch']} | "
                f"{fmt_time(last_epoch_s)}")
        state["history"].append(row)
        atomic_torch_save({"trainable": trainable_state(ENC), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                           "state": state, "tag": C.P7_TAG}, LATEST)
        atomic_json({"tag": C.P7_TAG, "epoch": state["epoch"], "best_f1": state["best_f1"]}, LATEST_META)
        pd.DataFrame(state["history"]).to_csv(LOG_CSV, index=False)
        if state["bad"] >= C.PATIENCE:
            log(f"early stopping: no dev improvement for {C.PATIENCE} evaluations")
            break

    pd.DataFrame(state["history"]).to_csv(LOG_CSV, index=False)
    RESULTS["training"] = {"history": state["history"], "best_dev_f1": state["best_f1"], "best_epoch": state["best_epoch"],
                           "train_seconds": time.time() - t_train0, "docs_per_s": docs_seen / max(t_docs, 1e-9),
                           "peak_gpu_mem_gb": (torch.cuda.max_memory_allocated() / 2 ** 30) if DEVICE.type == "cuda" else None}
    return state

if C.TRAIN:
    TRAIN_STATE = run_training()
else:
    TRAIN_STATE = None
    log("TRAIN=False -> skipping training")

cands = [p for p in (BEST,) if os.path.exists(p)]
if not cands:
    for d in walk_find("p7_best_meta.json", [INPUT_ROOT]):
        meta = read_json(os.path.join(d, "p7_best_meta.json"))
        if meta and meta.get("tag") == C.P7_TAG and os.path.exists(os.path.join(d, "p7_best.pt")):
            cands.append(os.path.join(d, "p7_best.pt"))
if cands:
    bl = torch.load(cands[0], map_location="cpu", weights_only=False)
    load_trainable_state(ENC, bl["trainable"])
    log(f"loaded best adapters from {cands[0]} (epoch {bl.get('epoch')}, dev F1@7 {bl.get('dev_f1', float('nan')):.4f})")
else:
    log("no trained adapters found - continuing with the zero-shot model")

train queries with gold in the pool: 817/827 | 103 steps/epoch x 8 epochs = 824 steps | warm-up 82
epoch 0 (zero-shot, LoRA at init)  dev P@7=0.3257 R@7=0.2765 F1@7=0.2991
[ep 1/8] step 10/103 | loss 2.7232 | in-batch acc 0.312 | gnorm 2.89 | lr 2.44e-05 | 9.7 docs/s | elapsed 0h05m00s
[ep 1/8] step 20/103 | loss 2.6506 | in-batch acc 0.325 | gnorm 2.30 | lr 4.88e-05 | 9.9 docs/s | elapsed 0h05m31s
[ep 1/8] step 30/103 | loss 2.6136 | in-batch acc 0.317 | gnorm 3.33 | lr 7.32e-05 | 9.9 docs/s | elapsed 0h06m03s
[ep 1/8] step 40/103 | loss 2.5283 | in-batch acc 0.325 | gnorm 5.44 | lr 9.76e-05 | 10.0 docs/s | elapsed 0h06m35s
[ep 1/8] step 50/103 | loss 2.4828 | in-batch acc 0.325 | gnorm 4.14 | lr 1.22e-04 | 10.0 docs/s | elapsed 0h07m06s
[ep 1/8] step 60/103 | loss 2.4539 | in-batch acc 0.333 | gnorm 2.59 | lr 1.46e-04 | 10.0 docs/s | elapsed 0h07m38s
[ep 1/8] step 70/103 | loss 2.4082 | in-batch acc 0.336 | gnorm 3.18 | lr 1.71e-04 | 10.0 docs/s | elapsed 0h08m10s
[ep 1/8] step 80/10

### 9. Dev evaluation, pooling analysis, hybrid analysis, failure analysis

In [9]:

# Validation and score fusion

dev = SP["dev"]
if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
raw_dev = encode_split_raw(ENC, dev)
RESULTS["efficiency_adapted"] = {
    "dev_docs": raw_dev["n_docs"], "dev_chunks": raw_dev["n_chunks"], "seconds": raw_dev["seconds"],
    "docs_per_s": raw_dev["n_docs"] / max(raw_dev["seconds"], 1e-9),
    "peak_gpu_mem_gb": (torch.cuda.max_memory_allocated() / 2 ** 30) if DEVICE.type == "cuda" else None,
    "note": "LoRA kept un-merged; merging B@A into W would remove the adapter overhead"}
log(f"[adapted] encoded dev in {fmt_time(raw_dev['seconds'])} ({RESULTS['efficiency_adapted']['docs_per_s']:.1f} docs/s)")

doc_pools = ("mean", "max", "first") + (("attn",) if C.DOC_POOL == "attn" else ())
GRID1 = dev_grid(ENC, raw_dev, dev, doc_pools=doc_pools)
log("[adapted] dev, every inference-time pooling variant (the model was TRAINED with "
    f"token_pool={C.TOKEN_POOL}, doc_pool={C.DOC_POOL}):")
log(GRID1.to_string(index=False))
RESULTS["adapted_grid"] = GRID1.to_dict("records")

cmp_ = {}
for ce in ("none", "pool"):
    Qe_, Ce_ = embed_from_raw(ENC, raw_dev, C.TOKEN_POOL, C.DOC_POOL, ce)
    cmp_[ce] = evaluate_dense(Qe_, Ce_, dev)[0]["F1"]
FINAL_CENTER = max(cmp_, key=cmp_.get)
log(f"centring on dev: {cmp_} -> using '{FINAL_CENTER}'")

Qe_dev, Ce_dev = embed_from_raw(ENC, raw_dev, C.TOKEN_POOL, C.DOC_POOL, FINAL_CENTER)
S_dense_dev = Qe_dev @ Ce_dev.T

# hybrid dense + TF-IDF (analysis; used for the submission only if USE_HYBRID)
hy_rows = []
for lam in (0.0, 0.05, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0, 2.0):
    mm = micro_prf(rank_topk(hybrid_scores(S_dense_dev, TFIDF_DEV, lam), dev.qids, dev.cids, C.K), dev.gold, dev.qids, C.K)
    hy_rows.append({"lambda_tfidf": lam, "P@7": round(mm["P"], 4), "R@7": round(mm["R"], 4), "F1@7": round(mm["F1"], 4)})
HY = pd.DataFrame(hy_rows)
log("[adapted] hybrid ranking  z(dense) + lambda * z(TF-IDF)  on dev:")
log(HY.to_string(index=False))
RESULTS["hybrid_dev"] = hy_rows
BEST_LAMBDA = float(HY.loc[HY["F1@7"].idxmax(), "lambda_tfidf"])
FINAL_LAMBDA = BEST_LAMBDA if C.USE_HYBRID else None

S_final_dev = S_dense_dev if FINAL_LAMBDA is None else hybrid_scores(S_dense_dev, TFIDF_DEV, FINAL_LAMBDA)
preds_dev = rank_topk(S_final_dev, dev.qids, dev.cids, C.K)
M_DEV = micro_prf(preds_dev, dev.gold, dev.qids, C.K)
log(f"\n>>> FINAL dev (submission configuration: token_pool={C.TOKEN_POOL}, doc_pool={C.DOC_POOL}, centre={FINAL_CENTER}, "
    f"hybrid={FINAL_LAMBDA}): P@7={M_DEV['P']:.4f}  R@7={M_DEV['R']:.4f}  F1@7={M_DEV['F1']:.4f}")
RESULTS["dev_final"] = {**M_DEV, "token_pool": C.TOKEN_POOL, "doc_pool": C.DOC_POOL, "center": FINAL_CENTER,
                        "hybrid_lambda": FINAL_LAMBDA}

# per-query failure analysis
PQ = per_query_table(S_final_dev, dev, preds_dev, C.K)
PQ["f1"] = 2 * PQ["hits"] / (C.K + PQ["n_gold"])
PQ.to_csv(os.path.join(OUT_DIR, "dev_per_query.csv"), index=False)
n_zero = int((PQ["hits"] == 0).sum())
_med = PQ.loc[PQ.first_hit_rank > 0, "first_hit_rank"].median()
log(f"queries with 0 hits in the top-7: {n_zero}/{len(PQ)} ({100 * n_zero / len(PQ):.1f}%) | "
    f"median rank of the first gold case: {int(_med) if _med == _med else 'n/a'}")
try:
    PQ["len_bucket"] = pd.qcut(PQ["q_tokens"], q=min(5, PQ["q_tokens"].nunique()), duplicates="drop")
except ValueError:
    PQ["len_bucket"] = "all"
PQ["gold_bucket"] = pd.cut(PQ["n_gold"], bins=[0, 1, 3, 6, 10**6], labels=["1", "2-3", "4-6", "7+"])
log("\nby query length (tokens):")
log(PQ.groupby("len_bucket", observed=True).agg(queries=("qid", "count"), recall=("recall", "mean"),
                                               precision=("precision", "mean"), zero_hit=("hits", lambda s: (s == 0).mean())).round(3).to_string())
log("\nby number of gold cases:")
log(PQ.groupby("gold_bucket", observed=True).agg(queries=("qid", "count"), recall=("recall", "mean"),
                                                precision=("precision", "mean"), zero_hit=("hits", lambda s: (s == 0).mean())).round(3).to_string())
np.savez_compressed(os.path.join(OUT_DIR, "dev_embeddings.npz"), Q=Qe_dev, C=Ce_dev,
                    qids=np.array(dev.qids), cids=np.array(dev.cids))        # reusable as a teacher signal in later experiments
del raw_dev
gc.collect()

[adapted] encoded dev in 0h01m34s (12.0 docs/s)
[adapted] dev, every inference-time pooling variant (the model was TRAINED with token_pool=mean, doc_pool=mean):
token_pool doc_pool center    P@7    R@7   F1@7
      mean     mean   none 0.3862 0.3279 0.3546
      mean     mean   pool 0.3814 0.3237 0.3502
      mean      max   pool 0.3269 0.2775 0.3002
      mean    first   pool 0.3196 0.2713 0.2935
      mean      max   none 0.3160 0.2682 0.2902
      mean    first   none 0.3136 0.2662 0.2879
      last     mean   pool 0.2506 0.2127 0.2301
      last     mean   none 0.2252 0.1912 0.2068
      last      max   pool 0.1235 0.1048 0.1134
      last      max   none 0.0944 0.0802 0.0867
      last    first   pool 0.0799 0.0678 0.0734
      last    first   none 0.0654 0.0555 0.0600
centring on dev: {'none': 0.35464146748193437, 'pool': 0.35019455252918286} -> using 'none'
[adapted] hybrid ranking  z(dense) + lambda * z(TF-IDF)  on dev:
 lambda_tfidf    P@7    R@7   F1@7
         0.00 0.3862 0.

9

### 10. Test predictions and submission files

In [10]:

# Test predictions

PRED_PATH = os.path.join(OUT_DIR, "prediction.json")
ZIP_PATH = os.path.join(OUT_DIR, "submission.zip")

if C.RUN_PREDICT:
    te = SP["test"]
    raw_te = encode_split_raw(ENC, te, need=(C.TOKEN_POOL,))
    Qe_te, Ce_te = embed_from_raw(ENC, raw_te, C.TOKEN_POOL, C.DOC_POOL, FINAL_CENTER)
    S_te = Qe_te @ Ce_te.T
    if FINAL_LAMBDA is not None:
        S_te = hybrid_scores(S_te, tfidf_scores([te.qtext[q] for q in te.qids], [te.ctext[c] for c in te.cids]), FINAL_LAMBDA)
    pred_te = rank_topk(S_te, te.qids, te.cids, C.K)

    pool_te = set(te.cids)
    for q in te.qids:
        p = pred_te[q]
        assert len(p) == C.K, (q, len(p))
        assert len(set(p)) == C.K, f"duplicate ids for query {q}"
        assert all(c in pool_te for c in p), f"id outside the test pool for query {q}"
        assert q not in p, f"query {q} retrieved itself"

    with open(PRED_PATH, "w", encoding="utf-8") as f:
        json.dump({q: pred_te[q] for q in te.qids}, f, indent=2)
    chk = json.load(open(PRED_PATH, encoding="utf-8"))
    assert list(chk.keys()) == te.qids and all(len(v) == C.K and len(set(v)) == C.K for v in chk.values())
    with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(PRED_PATH, arcname="prediction.json")
    np.savez_compressed(os.path.join(OUT_DIR, "test_embeddings.npz"), Q=Qe_te, C=Ce_te,
                        qids=np.array(te.qids), cids=np.array(te.cids))
    log(f"wrote {PRED_PATH}: {len(chk)} test queries x {C.K} ids | {ZIP_PATH}")
    if C.MAX_QUERIES or C.MAX_POOL:
        log("WARNING: this was a smoke run (MAX_QUERIES / MAX_POOL set) - the file covers only part of the test set. DO NOT SUBMIT IT.")
    log("example:", list(chk.items())[0])
    RESULTS["test"] = {"queries": len(chk), "pool": len(te.cids), "encode_seconds": raw_te["seconds"]}
    del raw_te
    gc.collect()
else:
    log("RUN_PREDICT=False -> no prediction file written")

wrote /kaggle/working/cs781_phase7_exp1_model2_lora_docpool/prediction.json: 237 test queries x 7 ids | /kaggle/working/cs781_phase7_exp1_model2_lora_docpool/submission.zip
example: ('0000021652', ['0000590455', '0001481813', '0000307495', '0001505859', '0001724804', '0001388203', '0000425869'])


### 11. Summary for the report

In [11]:

# Results summary

def _best_zero_shot():
    g = RESULTS.get("zero_shot_grid")
    if not g:
        return None
    return max(g, key=lambda r: r["F1@7"])

zs = _best_zero_shot()
rows = [{"system": "TF-IDF (classical baseline)", "P@7": RESULTS["tfidf_dev"]["P"], "R@7": RESULTS["tfidf_dev"]["R"],
         "F1@7": RESULTS["tfidf_dev"]["F1"], "trainable params": 0}]
if zs:
    rows.append({"system": f"Phase-1 zero-shot (best pooling: {zs['token_pool']}/{zs['doc_pool']}/centre={zs['center']})",
                 "P@7": zs["P@7"], "R@7": zs["R@7"], "F1@7": zs["F1@7"], "trainable params": 0})
rows.append({"system": f"Exp 1: Phase-1 + LoRA(r={C.LORA_R}) + doc pooling ({C.TOKEN_POOL}/{C.DOC_POOL}) + TF-IDF hard negatives",
             "P@7": M_DEV["P"], "R@7": M_DEV["R"], "F1@7": M_DEV["F1"], "trainable params": RESULTS["params"]["trainable"]})
SUMMARY = pd.DataFrame(rows)
log(SUMMARY.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

ea = RESULTS.get("efficiency_adapted", {})
log(f"\nmodel: {RESULTS['params']['base'] / 1e6:.1f}M frozen Phase-1 params + {RESULTS['params']['trainable'] / 1e6:.2f}M trainable "
    f"({100 * RESULTS['params']['trainable'] / RESULTS['params']['total']:.2f}%) | "
    f"inference {ea.get('docs_per_s', float('nan')):.1f} docs/s | peak GPU memory (inference) {ea.get('peak_gpu_mem_gb')} GB | "
    f"training peak {RESULTS.get('training', {}).get('peak_gpu_mem_gb')} GB")
RESULTS["summary"] = rows
RESULTS["total_runtime_s"] = time.time() - T_START

def _jsonable(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (set, tuple)):
        return list(o)
    return str(o)

with open(os.path.join(OUT_DIR, "results_exp1.json"), "w", encoding="utf-8") as f:
    json.dump(RESULTS, f, indent=1, default=_jsonable)
for fn in ("p7_best.pt", "p7_train_log.csv"):                       # the adapters are tiny: keep them next to the outputs
    src = os.path.join(CKPT_DIR, fn)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(OUT_DIR, fn))
log(f"\nartifacts in {OUT_DIR}: " + ", ".join(sorted(os.listdir(OUT_DIR))))
log(f"total runtime {fmt_time(time.time() - T_START)}")

                                                                       system    P@7    R@7   F1@7  trainable params
                                                  TF-IDF (classical baseline) 0.3995 0.3392 0.3669                 0
                      Phase-1 zero-shot (best pooling: mean/mean/centre=pool) 0.3354 0.2847 0.3079                 0
Exp 1: Phase-1 + LoRA(r=16) + doc pooling (mean/mean) + TF-IDF hard negatives 0.3862 0.3279 0.3546           1966080

model: 59.8M frozen Phase-1 params + 1.97M trainable (3.18%) | inference 12.0 docs/s | peak GPU memory (inference) 0.9831395149230957 GB | training peak 2.419900894165039 GB

artifacts in /kaggle/working/cs781_phase7_exp1_model2_lora_docpool: dev_embeddings.npz, dev_per_query.csv, p7_best.pt, p7_train_log.csv, prediction.json, results_exp1.json, submission.zip, test_embeddings.npz
total runtime 0h57m43s
